# Single Product Evidence Harness

This notebook runs one product through the retailer-scrapability-aware agentic loop.

Default output is now a compact `final_row.csv` plus markdown evidence packet. Detailed CSV dumps are disabled unless `PRODUCT_HARNESS_WRITE_DEBUG_CSVS=true`.

## Environment / kernel

Run `bash scripts/setup_in_project_venv.sh` once from the project root. Then select the Jupyter kernel `Product Evidence Harness (.venv)`. The notebook still inserts `src/` on `sys.path` for safety, but normal use should be through the project `.venv`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT

In [ ]:
from product_evidence_harness import (
    HarnessConfig,
    ProductEvidenceHarness,
    ProductQuery,
    SerpAPIConfig,
    configure_logging,
)

configure_logging('INFO')

## Configure one product

`retailer_name` is a preferred first evidence source. If that retailer is not scrape-usable/rich/exact, the harness escapes to other retailers in the same country, then global fallback.

In [ ]:
product = ProductQuery(
    row_id='CO-ML-0001',
    main_text='PUT PRODUCT TEXT HERE',
    country_code='CO',
    ean='',  # optional; keep as text
    retailer_name='Mercado Libre',  # optional preferred-first evidence source
)
product

In [ ]:
config = HarnessConfig.from_env(PROJECT_ROOT / '.env')
serp_config = SerpAPIConfig.from_env(
    country_code=product.country_code,
    language_code=product.language_code or 'es',
    env_file=PROJECT_ROOT / '.env',
)

harness = ProductEvidenceHarness(serp_config=serp_config, config=config)
trace = harness.run(product, return_trace=True)
trace.best_match.to_dict()

## Inspect output artifact packet

In [ ]:
row_dir = Path(config.output_dir) / product.row_id
print('Row artifact directory:', row_dir)
print('Expected files:')
for p in sorted(row_dir.iterdir()):
    print(' -', p.name)

In [ ]:
# Submission-ready one-row CSV
import pandas as pd
pd.read_csv(row_dir / 'final_row.csv', dtype=str).T

In [ ]:
# Human/LLM-readable master report
print((row_dir / 'report.md').read_text(encoding='utf-8')[:4000])

In [ ]:
# Compact machine trace
import json
trace_json = json.loads((row_dir / 'trace.json').read_text(encoding='utf-8'))
trace_json.keys()